In [1]:
import kagglehub
import pandas as pd

In [3]:
kagglehub.login()

In [2]:
# Pobranie danych
path = kagglehub.dataset_download("arannayavadebnath/ferrero-rocher-sales-dataset")

In [3]:
# Wczytanie danych
data = pd.read_csv(f"{path}/Ferrero_rocher/ferrero_rocher_sales_dataset.csv")

print(f"Wielkość danych: {data.shape}")

Wielkość danych: (20000, 12)


In [4]:
# Czyszczenie danych
data_clean = data.copy()
data_clean = data_clean.drop_duplicates()
data_clean['Date'] = pd.to_datetime(data_clean['Date'], origin='1899-12-30', unit='D')
data_clean = data_clean.dropna(subset=[col for col in data_clean.columns if data_clean[col].name != "Promotion"])
data_clean['Promotion'] = data_clean['Promotion'].fillna('No Promotion')

In [11]:
# Statystyki zbioru danych
print(f"Liczba wierszy: {len(data_clean)}")
print(f"Liczba kolumn: {len(data_clean.columns)}")

categorical_cols = data_clean.select_dtypes(include=['object', 'str']).columns
print("\nRozkład zmiennych kategorycznych:")

for col in categorical_cols:
    print(f"\n{col}:")
    counts = data_clean[col].value_counts()
    percentages = data_clean[col].value_counts(normalize=True) * 100
    for value, count in counts.items():
        print(f"  {value}: {count} ({percentages[value]:.2f}%)")

numeric_cols = data_clean.select_dtypes(include=['number']).columns

print("\nWartości MIN i MAX dla każdej kolumny numerycznej:")
for col in numeric_cols:
    print(f"{col}:")
    print(f"Min: {data_clean[col].min()}")
    print(f"Max: {data_clean[col].max()}")

print("\nStatystyki zmiennych numerycznych:")
print(data_clean[numeric_cols].describe())

Liczba wierszy: 20000
Liczba kolumn: 12

Rozkład zmiennych kategorycznych:

Store:
  Walmart: 2486 (12.43%)
  FairPrice(NTUC): 2446 (12.23%)
  Tesco: 2431 (12.16%)
  7-Eleven: 2381 (11.91%)
  Carrefour: 1639 (8.20%)
  Amazon Fresh : 1221 (6.11%)
  DMart: 810 (4.05%)
  Sobeys: 440 (2.20%)
  Loblaws: 433 (2.17%)
  Reliance Fresh: 433 (2.17%)
  Viva: 426 (2.13%)
  Kmart: 417 (2.08%)
  Costco: 414 (2.07%)
  Spencers: 402 (2.01%)
  Reliance Smart Bazar: 401 (2.00%)
  Spinneys: 401 (2.00%)
  Asda: 400 (2.00%)
  ShengSiong: 400 (2.00%)
  Woolworths: 399 (1.99%)
  Aldy: 393 (1.97%)
  Lulu Hypermarket: 392 (1.96%)
  BigW: 270 (1.35%)
  Flipkart Groceries: 180 (0.90%)
  Metro Cash & Carry : 142 (0.71%)
  Coles: 127 (0.64%)
  Zepto: 60 (0.30%)
  Swiggy Instamart: 30 (0.15%)
  Big Basket: 26 (0.13%)

Country:
  UK: 3010 (15.05%)
  Canada: 2958 (14.79%)
  USA: 2851 (14.26%)
  Singapore: 2846 (14.23%)
  UAE: 2818 (14.09%)
  Australia: 2777 (13.88%)
  India: 2740 (13.70%)

SKU:
  Ferrero Rocher T3: 4

In [6]:
# Podział danych na zbiory treningowy, walidacyjny i testowy
from sklearn.model_selection import train_test_split

train_data, temp_data = train_test_split(data_clean, test_size=0.2, random_state=42)
dev_data, test_data = train_test_split(temp_data, test_size=0.5, random_state=42)

print(f"Wielkość zbioru treningowego: {len(train_data)}")
print(f"Wielkość zbioru walidacyjnego: {len(dev_data)}")
print(f"Wielkość zbioru testowego: {len(test_data)}")

Wielkość zbioru treningowego: 16000
Wielkość zbioru walidacyjnego: 2000
Wielkość zbioru testowego: 2000


In [10]:
# Statystyki dla zbiorów treningowego, walidacyjnego i testowego

def print_subset_stats(name, df):
    print(f"Statystyki dla zbioru: {name}")

    numeric_cols = df.select_dtypes(include=['number']).columns
    print("\nStatystyki zmiennych numerycznych (mean, min, max, std, std, median):")
    stats = df[numeric_cols].agg(['mean', 'min', 'max', 'std', 'median'])
    print(stats)

    categorical_cols = df.select_dtypes(include=['object', 'str']).columns
    if len(categorical_cols) > 0:
        print("\nRozkład zmiennych kategorycznych:")
        for col in categorical_cols:
            print(f"\n{col}:")
            counts = df[col].value_counts()
            percentages = df[col].value_counts(normalize=True) * 100
            for value, count in counts.head(5).items():
                print(f"  {value}: {count} ({percentages[value]:.2f}%)")

print_subset_stats("treningowy", train_data)
print_subset_stats("walidacyjny", dev_data)
print_subset_stats("testowy", test_data)

Statystyki dla zbioru: treningowy

Statystyki zmiennych numerycznych (mean, min, max, std, std, median):
        Units Sold  Unit Price  Discount        Revenue   Margin %  \
mean    104.810312  481.965625  0.199672   40397.542469  32.506813   
min      10.000000  120.000000  0.000000     600.000000  25.000000   
max     199.000000  750.000000  0.500000  149250.000000  40.000000   
std      54.957392  225.888621  0.154506   31561.421730   4.630069   
median  105.000000  550.000000  0.150000   32465.000000  32.000000   

             Margin  
mean    13140.94328  
min       156.00000  
max     57622.50000  
std     10552.91759  
median  10351.95000  

Rozkład zmiennych kategorycznych:

Store:
  Walmart: 2011 (12.57%)
  FairPrice(NTUC): 1957 (12.23%)
  Tesco: 1952 (12.20%)
  7-Eleven: 1898 (11.86%)
  Carrefour: 1305 (8.16%)

Country:
  UK: 2441 (15.26%)
  Canada: 2360 (14.75%)
  Singapore: 2272 (14.20%)
  USA: 2263 (14.14%)
  UAE: 2248 (14.05%)

SKU:
  Ferrero Rocher T3: 3265 (20.41%)
  

In [8]:
# Normalizacja cech numerycznych
from sklearn.preprocessing import MinMaxScaler

numeric_cols = train_data.select_dtypes(include=['number']).columns
scaler = MinMaxScaler()

train_data_norm = train_data.copy()
dev_data_norm = dev_data.copy()
test_data_norm = test_data.copy()

train_data_norm[numeric_cols] = scaler.fit_transform(train_data[numeric_cols])

dev_data_norm[numeric_cols] = scaler.transform(dev_data[numeric_cols])
test_data_norm[numeric_cols] = scaler.transform(test_data[numeric_cols])

Przykładowe znormalizowane wartości (zbiór treningowy):
      Units Sold  Unit Price  Discount   Revenue  Margin %    Margin
5894    0.476190    0.365079       0.2  0.207871  0.400000  0.167210
3728    0.698413    0.682540       1.0  0.258661  0.866667  0.255505
8958    0.280423    0.682540       1.0  0.112513  0.933333  0.114863
7671    0.624339    0.365079       1.0  0.146653  0.733333  0.137611
5999    0.132275    0.682540       0.5  0.093088  0.133333  0.065118
